# Build a Contextual Retrieval based RAG System

## Install OpenAI, and LangChain dependencies

In [2]:
!pip install langchain==1.2.12
!pip install langchain-openai==1.1.11
!pip install langchain-community==0.4.1
!pip install jq==1.11.0
!pip install pymupdf==1.27.2

## Install Chroma Vector DB and LangChain wrapper

In [3]:
!pip install langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.0 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelemet

## Enter Open AI API Key

In [4]:

from getpass import getpass

OPENAI_KEY = getpass('Enter Open AI API Key: ')

Enter Open AI API Key: ··········


## Setup Environment Variables

In [5]:
import os

os.environ['OPENAI_API_KEY'] = OPENAI_KEY

### Open AI Embedding Models

LangChain enables us to access Open AI embedding models which include the newest models: a smaller and highly efficient `text-embedding-3-small` model, and a larger and more powerful `text-embedding-3-large` model.

In [6]:
from langchain_openai import OpenAIEmbeddings

# details here: https://openai.com/blog/new-embedding-models-and-api-updates
openai_embed_model = OpenAIEmbeddings(model='text-embedding-3-small')

## Loading and Processing the Data

### Get the dataset

In [7]:
# if you can't download using the following code
# go to https://drive.google.com/file/d/1aZxZejfteVuofISodUrY2CDoyuPLYDGZ download it
# manually upload it on colab
!gdown 1aZxZejfteVuofISodUrY2CDoyuPLYDGZ

Downloading...
From: https://drive.google.com/uc?id=1aZxZejfteVuofISodUrY2CDoyuPLYDGZ
To: /content/rag_docs.zip
100% 5.92M/5.92M [00:00<00:00, 54.7MB/s]


In [8]:
!unzip rag_docs.zip

Archive:  rag_docs.zip
   creating: rag_docs/
  inflating: rag_docs/attention_paper.pdf  
  inflating: rag_docs/cnn_paper.pdf  
  inflating: rag_docs/resnet_paper.pdf  
  inflating: rag_docs/vision_transformer.pdf  
  inflating: rag_docs/wikidata_rag_demo.jsonl  


### Load and Process JSON Documents

In [10]:
from langchain_community.document_loaders import JSONLoader

loader = JSONLoader(
    file_path="./rag_docs/wikidata_rag_demo.jsonl",
    jq_schema=".",
    text_content=False,
    json_lines=True,
)
wiki_docs = loader.load()

In [11]:
len(wiki_docs)

1801

In [12]:
wiki_docs[3]

Document(metadata={'source': '/content/rag_docs/wikidata_rag_demo.jsonl', 'seq_num': 4}, page_content='{"id": "71548", "title": "Chi-square distribution", "paragraphs": ["In probability theory and statistics, the chi-square distribution (also chi-squared or formula_1\\u00a0 distribution) is one of the most widely used theoretical probability distributions. Chi-square distribution with formula_2 degrees of freedom is written as formula_3. It is a special case of gamma distribution.", "Chi-square distribution is primarily used in statistical significance tests and confidence intervals. It is useful, because it is relatively easy to show that certain probability distributions come close to it, under certain conditions. One of these conditions is that the null hypothesis must be true. Another one is that the different random variables (or observations) must be independent of each other."]}')

In [13]:
import json
from langchain_core.documents import Document

wiki_docs_processed = []

for doc in wiki_docs:
    doc = json.loads(doc.page_content)
    metadata = {
        "title": doc["title"],
        "id": doc["id"],
        "source": "Wikipedia",
        "page": 1,
    }
    data = " ".join(doc["paragraphs"])
    wiki_docs_processed.append(Document(page_content=data, metadata=metadata))

In [14]:
wiki_docs_processed[3]

Document(metadata={'title': 'Chi-square distribution', 'id': '71548', 'source': 'Wikipedia', 'page': 1}, page_content='In probability theory and statistics, the chi-square distribution (also chi-squared or formula_1\xa0 distribution) is one of the most widely used theoretical probability distributions. Chi-square distribution with formula_2 degrees of freedom is written as formula_3. It is a special case of gamma distribution. Chi-square distribution is primarily used in statistical significance tests and confidence intervals. It is useful, because it is relatively easy to show that certain probability distributions come close to it, under certain conditions. One of these conditions is that the null hypothesis must be true. Another one is that the different random variables (or observations) must be independent of each other.')

### Load and Process PDF documents

#### Create Chunk Contexts for Contextual Retrieval

![](https://i.imgur.com/LRhKHzk.png)

In [15]:
from langchain_openai import ChatOpenAI

chatgpt = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

In [17]:
# create chunk context generation chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


def generate_chunk_context(document, chunk):

    chunk_process_prompt = """You are an AI assistant specializing in research paper analysis.
                            Your task is to provide brief, relevant context for a chunk of text
                            based on the following research paper.

                            Here is the research paper:
                            <paper>
                            {paper}
                            </paper>

                            Here is the chunk we want to situate within the whole document:
                            <chunk>
                            {chunk}
                            </chunk>

                            Provide a concise context (3-4 sentences max) for this chunk,
                            considering the following guidelines:

                            - Give a short succinct context to situate this chunk within the overall document
                            for the purposes of improving search retrieval of the chunk.
                            - Answer only with the succinct context and nothing else.
                            - Context should be mentioned like 'Focuses on ....'
                            do not mention 'this chunk or section focuses on...'

                            Context:
                        """

    prompt_template = ChatPromptTemplate.from_template(chunk_process_prompt)

    agentic_chunk_chain = (prompt_template
                                |
                            chatgpt
                                |
                            StrOutputParser())

    context = agentic_chunk_chain.invoke({'paper': document, 'chunk': chunk})

    return context

In [19]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import uuid


def create_contextual_chunks(file_path, chunk_size=3500, chunk_overlap=0):

    print('Loading pages:', file_path)
    loader = PyMuPDFLoader(file_path)
    doc_pages = loader.load()

    print('Chunking pages:', file_path)
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size,
                                              chunk_overlap=chunk_overlap)
    doc_chunks = splitter.split_documents(doc_pages)

    print('Generating contextual chunks:', file_path)
    original_doc = '\n'.join([doc.page_content for doc in doc_chunks])
    contextual_chunks = []
    for chunk in doc_chunks:
        chunk_content = chunk.page_content
        chunk_metadata = chunk.metadata
        chunk_metadata_upd = {
            'id': str(uuid.uuid4()),
            'page': chunk_metadata['page'],
            'source': chunk_metadata['source'],
            'title': chunk_metadata['source'].split('/')[-1]
        }
        context = generate_chunk_context(original_doc, chunk_content)
        contextual_chunks.append(Document(page_content=context+'\n'+chunk_content,
                                          metadata=chunk_metadata_upd))
    print('Finished processing:', file_path)
    print()
    return contextual_chunks

In [20]:
from glob import glob

pdf_files = glob('./rag_docs/*.pdf')
pdf_files

['./rag_docs/resnet_paper.pdf',
 './rag_docs/attention_paper.pdf',
 './rag_docs/vision_transformer.pdf',
 './rag_docs/cnn_paper.pdf']

In [21]:
paper_docs = []
for fp in pdf_files:
    paper_docs.extend(create_contextual_chunks(file_path=fp, chunk_size=3500))

Loading pages: ./rag_docs/resnet_paper.pdf
Chunking pages: ./rag_docs/resnet_paper.pdf
Generating contextual chunks: ./rag_docs/resnet_paper.pdf
Finished processing: ./rag_docs/resnet_paper.pdf

Loading pages: ./rag_docs/attention_paper.pdf
Chunking pages: ./rag_docs/attention_paper.pdf
Generating contextual chunks: ./rag_docs/attention_paper.pdf
Finished processing: ./rag_docs/attention_paper.pdf

Loading pages: ./rag_docs/vision_transformer.pdf
Chunking pages: ./rag_docs/vision_transformer.pdf
Generating contextual chunks: ./rag_docs/vision_transformer.pdf
Finished processing: ./rag_docs/vision_transformer.pdf

Loading pages: ./rag_docs/cnn_paper.pdf
Chunking pages: ./rag_docs/cnn_paper.pdf
Generating contextual chunks: ./rag_docs/cnn_paper.pdf
Finished processing: ./rag_docs/cnn_paper.pdf



In [22]:
len(paper_docs)

79

In [23]:
paper_docs[0]

Document(metadata={'id': '6fd61838-2fd2-476c-9fac-55338705ba58', 'page': 0, 'source': './rag_docs/resnet_paper.pdf', 'title': 'resnet_paper.pdf'}, page_content='Focuses on the introduction of a residual learning framework to facilitate the training of deeper neural networks, addressing challenges such as vanishing gradients and degradation of accuracy with increased depth. It highlights the empirical success of residual networks on the ImageNet dataset and their application in various visual recognition tasks, including significant improvements in object detection performance.\nDeep Residual Learning for Image Recognition\nKaiming He\nXiangyu Zhang\nShaoqing Ren\nJian Sun\nMicrosoft Research\n{kahe, v-xiangz, v-shren, jiansun}@microsoft.com\nAbstract\nDeeper neural networks are more difﬁcult to train. We\npresent a residual learning framework to ease the training\nof networks that are substantially deeper than those used\npreviously. We explicitly reformulate the layers as learn-\ning 

### Combine all document chunks in one list

In [24]:
len(wiki_docs_processed)

1801

In [25]:
total_docs = wiki_docs_processed + paper_docs
len(total_docs)

1880

## Index Document Chunks and Embeddings in Vector DB

Here we initialize a connection to a Chroma vector DB client, and also we want to save to disk, so we simply initialize the Chroma client and pass the directory where we want the data to be saved to.

In [26]:
from langchain_chroma import Chroma

# create vector DB of docs and embeddings - takes < 30s on Colab
chroma_db = Chroma.from_documents(documents=total_docs,
                                  collection_name='my_context_db',
                                  embedding=openai_embed_model,
                                  # need to set the distance function to cosine else it uses euclidean by default
                                  # check https://docs.trychroma.com/guides#changing-the-distance-function
                                  collection_metadata={"hnsw:space": "cosine"},
                                  persist_directory="./my_context_db")

### Load Vector DB from disk

This is just to show once you have a vector database on disk you can just load and create a connection to it anytime

In [27]:
# load from disk
chroma_db = Chroma(persist_directory="./my_context_db",
                   collection_name='my_context_db',
                   embedding_function=openai_embed_model)

In [28]:
chroma_db

### Semantic Similarity based Retrieval

We use simple cosine similarity here and retrieve the top 5 similar documents based on the user input query

In [29]:
similarity_retriever = chroma_db.as_retriever(search_type="similarity",
                                              search_kwargs={"k": 5})

In [30]:
from IPython.display import display, Markdown

def display_docs(docs):
    for doc in docs:
        print('Metadata:', doc.metadata)
        print('Content Brief:')
        display(Markdown(doc.page_content[:1000]))
        print()

In [31]:
query = "what is machine learning?"
top_docs = similarity_retriever.invoke(query)
display_docs(top_docs)

Metadata: {'title': 'Machine learning', 'source': 'Wikipedia', 'id': '564928', 'page': 1}
Content Brief:


Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It is a subfield of computer science. The idea came from work in artificial intelligence. Machine learning explores the study and construction of algorithms which can learn and make predictions on data. Such algorithms follow programmed instructions, but can also make predictions or decisions based on data. They build a model from sample inputs. Machine learning is done where designing and programming explicit algorithms cannot be done. Examples include spam filtering, detection of network intruders or malicious insiders working towards a data breach, optical character recognition (OCR), search engines and computer vision.


Metadata: {'page': 1, 'id': '359370', 'title': 'Supervised learning', 'source': 'Wikipedia'}
Content Brief:


In machine learning, supervised learning is the task of inferring a function from labelled training data. The results of the training are known beforehand, the system simply learns how to get to these results correctly. Usually, such systems work with vectors. They get the training data and the result of the training as two vectors and produce a "classifier". Usually, the system uses inductive reasoning to generalize the training data.


Metadata: {'title': 'Deep learning', 'source': 'Wikipedia', 'page': 1, 'id': '663523'}
Content Brief:


Deep learning (also called deep structured learning or hierarchical learning) is a kind of machine learning, which is mostly used with certain kinds of neural networks. As with other kinds of machine-learning, learning sessions can be unsupervised, semi-supervised, or supervised. In many cases, structures are organised so that there is at least one intermediate layer (or hidden layer), between the input layer and the output layer. Certain tasks, such as as recognizing and understanding speech, images or handwriting, is easy to do for humans. However, for a computer, these tasks are very difficult to do. In a multi-layer neural network (having more than two layers), the information processed will become more abstract with each added layer. Deep learning models are inspired by information processing and communication patterns in biological nervous systems; they are different from the structural and functional properties of biological brains (especially the human brain) in many ways, whic


Metadata: {'id': '6360', 'title': 'Artificial intelligence', 'page': 1, 'source': 'Wikipedia'}
Content Brief:


Artificial intelligence (AI) is the ability of a computer program or a machine to think and learn. It is also a field of study which tries to make computers "smart". They work on their own without being encoded with commands. John McCarthy came up with the name "Artificial Intelligence" in 1955. In general use, the term "artificial intelligence" means a programme which mimics human cognition. At least some of the things we associate with other minds, such as learning and problem solving can be done by computers, though not in the same way as we do. Andreas Kaplan and Michael Haenlein define AI as a system’s ability to correctly interpret external data, to learn from such data, and to use those learnings to achieve specific goals and tasks through flexible adaptation. An ideal (perfect) intelligent machine is a flexible agent which perceives its environment and takes actions to maximize its chance of success at some goal or objective. As machines become increasingly capable, mental facu


Metadata: {'page': 1, 'id': '44742', 'source': 'Wikipedia', 'title': 'Artificial neural network'}
Content Brief:


A neural network (also called an ANN or an artificial neural network) is a sort of computer software, inspired by biological neurons. Biological brains are capable of solving difficult problems, but each neuron is only responsible for solving a very small part of the problem. Similarly, a neural network is made up of cells that work together to produce a desired result, although each individual cell is only responsible for solving a small part of the problem. This is one method for creating artificially intelligent programs. Neural networks are an example of machine learning, where a program can change as it learns to solve a problem. A neural network can be trained and improved with each example, but the larger the neural network, the more examples it needs to perform well—often needing millions or billions of examples in the case of deep learning. There are two ways to think of a neural network. First is like a human brain. Second is like a mathematical equation.

In [32]:
query = "what is the difference between transformers and vision transformers?"
top_docs = similarity_retriever.invoke(query)
display_docs(top_docs)

Metadata: {'page': 7, 'source': './rag_docs/vision_transformer.pdf', 'id': 'd1c5e7ce-82ff-4fda-85d5-f44b0231f052', 'title': 'vision_transformer.pdf'}
Content Brief:


Focuses on a controlled scaling study of various models, including ResNets and Vision Transformers, evaluating their transfer performance from the JFT-300M dataset. It highlights the performance versus pre-training cost, revealing that Vision Transformers outperform ResNets in terms of compute efficiency and suggesting potential for further scaling efforts. Additionally, it discusses the performance of hybrid models in comparison to pure Vision Transformers.
Published as a conference paper at ICLR 2021
4.4
SCALING STUDY
We perform a controlled scaling study of different models by evaluating transfer performance from
JFT-300M. In this setting data size does not bottleneck the models’ performances, and we assess
performance versus pre-training cost of each model. The model set includes: 7 ResNets, R50x1,
R50x2 R101x1, R152x1, R152x2, pre-trained for 7 epochs, plus R152x2 and R200x3 pre-trained
for 14 epochs; 6 Vision Transformers, ViT-B/32, B/16, L/32, L/16, pre-trained for 7 epochs, plu


Metadata: {'source': './rag_docs/vision_transformer.pdf', 'page': 0, 'id': '5f9f481a-ccd7-4aac-9da7-ff4b981825f0', 'title': 'vision_transformer.pdf'}
Content Brief:


Focuses on the introduction of the Vision Transformer (ViT) model, which applies a standard Transformer architecture directly to image classification tasks by treating image patches as tokens. It highlights the limitations of traditional convolutional neural networks (CNNs) in computer vision and presents evidence that a pure Transformer can achieve competitive performance on various image recognition benchmarks when pre-trained on large datasets.
Published as a conference paper at ICLR 2021
AN IMAGE IS WORTH 16X16 WORDS:
TRANSFORMERS FOR IMAGE RECOGNITION AT SCALE
Alexey Dosovitskiy∗,†, Lucas Beyer∗, Alexander Kolesnikov∗, Dirk Weissenborn∗,
Xiaohua Zhai∗, Thomas Unterthiner, Mostafa Dehghani, Matthias Minderer,
Georg Heigold, Sylvain Gelly, Jakob Uszkoreit, Neil Houlsby∗,†
∗equal technical contribution, †equal advising
Google Research, Brain Team
{adosovitskiy, neilhoulsby}@google.com
ABSTRACT
While the Transformer architecture has become the de-facto standard for natural
language pr


Metadata: {'title': 'vision_transformer.pdf', 'source': './rag_docs/vision_transformer.pdf', 'page': 2, 'id': 'a728a170-d292-47c6-b14a-b4665c56e813'}
Content Brief:


Focuses on the architecture and methodology of the Vision Transformer (ViT), detailing how images are processed by splitting them into patches, embedding them, and utilizing a standard Transformer encoder for image classification tasks. It describes the model's design principles, including the use of position embeddings and the integration of a classification token, while referencing foundational work in Transformer architecture.
Published as a conference paper at ICLR 2021
Transformer Encoder
MLP 
Head
Vision Transformer (ViT)
*
Linear Projection of Flattened Patches
* Extra learnable
     [ cl ass]  embedding
1
2
3
4
5
6
7
8
9
0
Patch + Position 
Embedding
Class
Bird
Ball
Car
...
Embedded 
Patches
Multi-Head 
Attention
Norm
MLP
Norm
+
L x
+
Transformer Encoder
Figure 1: Model overview. We split an image into ﬁxed-size patches, linearly embed each of them,
add position embeddings, and feed the resulting sequence of vectors to a standard Transformer
encoder. In order to perform classiﬁ


Metadata: {'id': '51a0f506-eb8c-423f-876c-5ea1f04eb870', 'page': 1, 'source': './rag_docs/vision_transformer.pdf', 'title': 'vision_transformer.pdf'}
Content Brief:


Focuses on the performance of the Vision Transformer (ViT) in image recognition tasks, highlighting its ability to achieve state-of-the-art results when pre-trained on large datasets. It contrasts the inductive biases of convolutional neural networks (CNNs) with the advantages of large-scale training for ViT, demonstrating its effectiveness on various benchmarks. Additionally, it references related work in the field, particularly models that integrate self-attention with CNNs.
Published as a conference paper at ICLR 2021
inherent to CNNs, such as translation equivariance and locality, and therefore do not generalize well
when trained on insufﬁcient amounts of data.
However, the picture changes if the models are trained on larger datasets (14M-300M images). We
ﬁnd that large scale training trumps inductive bias. Our Vision Transformer (ViT) attains excellent
results when pre-trained at sufﬁcient scale and transferred to tasks with fewer datapoints. When
pre-trained on the public ImageNe


Metadata: {'source': './rag_docs/vision_transformer.pdf', 'page': 7, 'title': 'vision_transformer.pdf', 'id': '40b199c2-6cba-4842-8b51-e81bdf4b8131'}
Content Brief:


Focuses on the behavior of attention mechanisms in the Vision Transformer (ViT), highlighting how attention distances vary across layers and the implications of localized attention in hybrid models that incorporate convolutional networks. It also discusses the relationship between attention distance and network depth, indicating that deeper layers attend to semantically relevant regions for classification.
have consistently small attention distances in the low layers. This highly localized attention is
less pronounced in hybrid models that apply a ResNet before the Transformer (Figure 7, right),
suggesting that it may serve a similar function as early convolutional layers in CNNs. Further, the
attention distance increases with network depth. Globally, we ﬁnd that the model attends to image
regions that are semantically relevant for classiﬁcation (Figure 6).
4.6
SELF-SUPERVISION
Transformers show impressive performance on NLP tasks. However, much of their success stems
not only from the

## Build the RAG Pipeline

In [33]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = """You are an assistant who is an expert in question-answering tasks.
                Answer the following question using only the following pieces of retrieved context.
                If the answer is not in the context, do not make up answers, just say that you don't know.
                Keep the answer detailed and well formatted based on the information from the context.

                Question:
                {question}

                Context:
                {context}

                Answer:
            """

rag_prompt_template = ChatPromptTemplate.from_template(rag_prompt)

In [34]:
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

chatgpt = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_rag_chain = (
    {
        "context": (similarity_retriever
                      |
                    format_docs),
        "question": RunnablePassthrough()
    }
      |
    rag_prompt_template
      |
    chatgpt
)

In [35]:
from IPython.display import display, Markdown

query = "What is machine learning?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

Machine learning is a subfield of computer science that provides computers with the ability to learn without being explicitly programmed. The concept was introduced by Arthur Samuel in 1959 and is rooted in artificial intelligence. Machine learning focuses on the study and construction of algorithms that can learn from data and make predictions or decisions based on that data. These algorithms follow programmed instructions but can also adapt and improve their performance by building models from sample inputs.

Machine learning is particularly useful in scenarios where designing and programming explicit algorithms is impractical. Common applications include spam filtering, detecting network intruders or malicious insiders, optical character recognition (OCR), search engines, and computer vision.

Within machine learning, there are different types of learning approaches, such as supervised learning, where a function is inferred from labeled training data. In this case, the system learns to produce correct results based on known outcomes, typically using vectors for training data and results to create a "classifier." Inductive reasoning is often employed to generalize from the training data.

Additionally, deep learning is a specialized area of machine learning that utilizes neural networks, particularly those with multiple layers (known as multi-layer neural networks). Deep learning is effective for complex tasks like speech recognition, image understanding, and handwriting recognition, which are challenging for computers but relatively easy for humans. These models are inspired by the information processing patterns of biological nervous systems, although they differ significantly from the structural and functional properties of human brains.

In [36]:
query = "What is a CNN?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

A CNN, or Convolutional Neural Network, is a specialized type of Artificial Neural Network (ANN) primarily used for image-driven pattern recognition tasks. CNNs are designed to process data with a grid-like topology, such as images, and they excel in identifying patterns and features within visual data.

### Key Characteristics of CNNs:

1. **Three-Dimensional Neuron Organization**:
   - CNNs consist of neurons organized into three dimensions: height, width, and depth. The depth refers to the third dimension of an activation volume, which is different from the total number of layers in the network.

2. **Layer Types**:
   - CNNs are composed of three main types of layers:
     - **Convolutional Layers**: These layers apply filters to the input data to extract features. Each neuron in a convolutional layer connects to a small region of the input, allowing the network to learn spatial hierarchies of features.
     - **Pooling Layers**: These layers reduce the spatial dimensions of the input volume, helping to decrease the computational load and control overfitting.
     - **Fully-Connected Layers**: These layers connect every neuron in one layer to every neuron in the next layer, typically used at the end of the network to produce the final output.

3. **Architecture**:
   - A typical CNN architecture involves stacking multiple convolutional layers followed by pooling layers. This stacking allows the network to learn increasingly complex features of the input data.

4. **Functionality**:
   - The input layer holds the pixel values of the image, while the convolutional layers compute outputs based on local regions of the input. The final output layer condenses the information into class scores, indicating the likelihood of the input belonging to each class.

5. **Applications**:
   - CNNs are particularly effective in tasks such as image classification, object detection, and other visual recognition tasks due to their ability to automatically learn and extract relevant features from images.

In summary, CNNs are a powerful tool in the field of machine learning, especially for applications involving image data, due to their unique architecture and ability to learn spatial hierarchies of features.

In [37]:
query = "How is a resnet better than a CNN?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

A ResNet (Residual Network) is considered better than a traditional CNN (Convolutional Neural Network) primarily due to its architectural innovations that address optimization challenges associated with deeper networks. Here are the key advantages of ResNets over standard CNNs:

1. **Residual Learning Framework**: ResNets utilize a residual learning framework that allows the network to learn residual mappings instead of the original unreferenced mappings. This is achieved through the introduction of shortcut connections that skip one or more layers. The formulation of the network as \( F(x) + x \) (where \( F(x) \) is the residual mapping) makes it easier to optimize, especially as the network depth increases.

2. **Overcoming Degradation Problem**: Traditional CNNs often suffer from the degradation problem, where adding more layers leads to higher training errors. In contrast, ResNets can maintain or even improve performance with increased depth. For instance, a 34-layer ResNet outperforms an 18-layer ResNet by 2.8% in terms of validation error, demonstrating that deeper networks can be effectively trained without the degradation issues faced by plain networks.

3. **Lower Training Errors**: Empirical results show that ResNets exhibit considerably lower training errors compared to their plain counterparts. For example, the 34-layer ResNet has a top-1 error of 25.03%, while the 34-layer plain network has a top-1 error of 28.54%. This indicates that ResNets are more effective at learning from data as they scale in depth.

4. **Architectural Efficiency**: ResNets achieve significant accuracy gains without a corresponding increase in computational complexity. For instance, a 152-layer ResNet has lower complexity (11.3 billion FLOPs) compared to VGG-16 (15.3 billion FLOPs), while still achieving superior performance.

5. **Generalization Performance**: ResNets have demonstrated strong generalization performance across various tasks, including object detection and segmentation, outperforming traditional architectures like VGG-16 in competitive benchmarks.

In summary, the introduction of residual connections in ResNets allows for easier optimization of deeper networks, leading to lower training errors, better performance, and greater architectural efficiency compared to traditional CNNs.

In [38]:
query = "What is NLP and its relation to linguistics?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

Natural Language Processing (NLP) is a field within Artificial Intelligence that focuses on enabling computers to automatically understand and generate human languages. The term "Natural Language" specifically refers to human languages, distinguishing them from programming languages.

NLP is closely related to linguistics, as it involves the study of language structure, meaning, and usage. The goal of NLP is to create systems that can process and analyze large amounts of natural language data, allowing for tasks such as language translation, sentiment analysis, and speech recognition. By leveraging linguistic principles, NLP aims to improve the interaction between humans and machines, making it possible for computers to comprehend and produce language in a way that is meaningful to users.

In [39]:
query = "What is the difference between AI, ML and DL?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

The difference between AI, ML, and DL can be summarized as follows:

### Artificial Intelligence (AI)
- **Definition**: AI refers to the ability of a computer program or machine to think and learn, mimicking human cognition. It encompasses a broad range of technologies and applications aimed at making machines "smart."
- **Functionality**: AI systems can interpret external data, learn from it, and use those learnings to achieve specific goals through flexible adaptation. The term has evolved, and tasks once considered AI, like optical character recognition, are now seen as routine technologies.

### Machine Learning (ML)
- **Definition**: ML is a subfield of AI that focuses on the study and construction of algorithms that allow computers to learn from data without being explicitly programmed. It emerged from the broader field of artificial intelligence.
- **Functionality**: ML algorithms build models from sample inputs and can make predictions or decisions based on data. It is particularly useful in scenarios where designing explicit algorithms is impractical, such as spam filtering and computer vision.

### Deep Learning (DL)
- **Definition**: DL is a specialized subset of machine learning that primarily uses neural networks with multiple layers (deep neural networks) to process data. It is often referred to as deep structured learning or hierarchical learning.
- **Functionality**: In deep learning, information is processed through multiple layers, with each layer extracting increasingly abstract features from the data. This approach is particularly effective for complex tasks like speech and image recognition, which are challenging for traditional algorithms.

In summary, AI is the overarching field that includes both ML and DL. ML is a specific approach within AI that enables learning from data, while DL is a further specialization of ML that utilizes deep neural networks to handle complex data processing tasks.

In [40]:
query = "What is the difference between transformers and vision transformers?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

The difference between transformers and vision transformers primarily lies in their application and the way they process input data.

1. **Architecture and Input Handling**:
   - **Transformers**: Originally designed for natural language processing (NLP), transformers process sequences of tokens (words) using self-attention mechanisms. They take a 1D sequence of embeddings as input, where each token represents a word or part of a word.
   - **Vision Transformers (ViT)**: Vision transformers adapt the transformer architecture for image classification tasks by treating image patches as tokens. An image is divided into fixed-size patches, which are then flattened and linearly embedded into a sequence of vectors. This sequence is fed into a standard transformer encoder, allowing the model to process the image as a sequence of patch embeddings.

2. **Positional Information**:
   - In traditional transformers, positional embeddings are added to the token embeddings to retain the order of words in a sentence.
   - In vision transformers, similar positional embeddings are added to the patch embeddings to maintain spatial information about the image, allowing the model to understand the layout and relationships between different parts of the image.

3. **Attention Mechanism**:
   - Both transformers and vision transformers utilize self-attention, but in vision transformers, this mechanism allows the model to integrate information across the entire image, even in the early layers. This is different from convolutional neural networks (CNNs), which typically have a more localized receptive field.

4. **Performance and Efficiency**:
   - Vision transformers have been shown to outperform traditional convolutional networks (like ResNets) in terms of compute efficiency when pre-trained on large datasets. They require significantly less computational resources to achieve comparable or superior performance on image classification tasks.

5. **Inductive Bias**:
   - Traditional CNNs have inherent inductive biases such as translation equivariance and locality, which help them generalize well on image data. Vision transformers, while lacking these biases, can achieve competitive performance when trained on sufficiently large datasets, demonstrating that large-scale training can outweigh the benefits of these biases.

In summary, while both transformers and vision transformers share a common architectural foundation, they differ significantly in their input processing, the nature of the data they handle, and their application domains. Vision transformers specifically adapt the transformer model for effective image classification by treating image patches as tokens and leveraging self-attention to capture global information across the image.

In [41]:
query = "How is self-attention important in transformers?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

Self-attention is crucial in transformers for several reasons, particularly in the context of the Vision Transformer (ViT) and natural language processing (NLP) tasks. Here are the key points regarding its importance:

1. **Global Information Integration**: Self-attention allows the model to integrate information across the entire input, whether it be an image or a sentence. This capability is particularly beneficial in the Vision Transformer, where it enables the model to attend to various regions of an image, even in the lowest layers. This means that the model can capture long-range dependencies and relationships between different parts of the input, which is essential for tasks like classification.

2. **Attention Distance**: The concept of "attention distance" is significant in understanding how self-attention operates. In the ViT, attention distances vary across layers, with deeper layers attending to semantically relevant regions for classification. This indicates that as the model goes deeper, it becomes more adept at focusing on important features that contribute to the final output.

3. **Localized vs. Global Attention**: In the lower layers of the ViT, attention is highly localized, meaning that the model focuses on small regions of the input. This behavior is less pronounced in hybrid models that combine convolutional networks with transformers, suggesting that localized attention in early layers serves a similar purpose to early convolutional layers in CNNs. As the model depth increases, the attention becomes more global, allowing for a broader understanding of the input.

4. **Performance in NLP**: In NLP tasks, self-attention mechanisms enable transformers to track long-distance dependencies within sentences. For example, attention heads can learn to connect verbs with their relevant phrases, demonstrating the model's ability to capture complex relationships in language.

5. **Scalability and Efficiency**: Self-attention mechanisms contribute to the scalability of transformers. While naive applications of self-attention to images would be computationally expensive, various approximations and adaptations (like local multi-head attention) have been developed to make it feasible for larger inputs. This adaptability allows transformers to perform effectively on large datasets, achieving state-of-the-art results in various benchmarks.

In summary, self-attention is vital in transformers as it facilitates global information integration, allows for varying attention distances across layers, and enhances the model's ability to capture complex relationships in both images and language, all while maintaining scalability and efficiency.

In [43]:
query = "How does a resnet work?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

A ResNet, or Residual Network, is a type of deep neural network architecture designed to address the degradation problem that occurs when training very deep networks. The key innovation of ResNets is the introduction of residual learning through the use of shortcut connections.

### How ResNets Work:

1. **Residual Mapping**:
   - Instead of learning the desired underlying mapping \( H(x) \) directly, ResNets learn a residual mapping \( F(x) = H(x) - x \). This means that the network is trained to predict the difference between the desired output and the input, rather than the output itself.
   - The original mapping can then be expressed as \( H(x) = F(x) + x \). This formulation allows the network to focus on learning the residuals, which is often easier than learning the full mapping.

2. **Shortcut Connections**:
   - ResNets utilize shortcut connections that skip one or more layers. These connections perform identity mapping, meaning they add the input \( x \) directly to the output of the stacked layers. This addition does not introduce extra parameters or computational complexity.
   - The shortcut connections help to mitigate issues like vanishing gradients, making it easier to train deeper networks.

3. **Architecture**:
   - ResNets can be constructed with various depths, such as 18, 34, 50, 101, and even 152 layers. Each of these architectures incorporates multiple residual blocks, which consist of convolutional layers followed by the addition of the input via shortcut connections.
   - For example, a 34-layer ResNet has the same number of parameters as a plain 34-layer network but exhibits significantly lower training error and better generalization performance.

4. **Training Process**:
   - ResNets are trained using standard techniques such as Stochastic Gradient Descent (SGD) with backpropagation. They can be trained end-to-end without modifications to existing optimization algorithms.
   - The training process includes techniques like batch normalization and specific learning rate schedules to enhance convergence.

5. **Performance**:
   - Empirical results show that ResNets outperform traditional plain networks, especially as the depth increases. For instance, a 34-layer ResNet can achieve better accuracy than an 18-layer ResNet, demonstrating that deeper networks can be effectively optimized using the residual learning framework.

In summary, ResNets leverage residual learning and shortcut connections to facilitate the training of very deep networks, overcoming the degradation problem typically associated with increased depth in neural networks. This architecture has proven to be highly effective in various image classification tasks, including those on datasets like ImageNet and CIFAR-10.

In [44]:
query = "What is LangGraph?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

I don't know.

In [45]:
query = "What is an Agentic AI System?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

The context provided does not contain specific information about an "Agentic AI System." Therefore, I don't know what an Agentic AI System is.

In [46]:
query = "What is LangChain?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

I don't know.

# Build a RAG System with Sources

In [47]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = """You are an assistant who is an expert in question-answering tasks.
                Answer the following question using only the following pieces of retrieved context.
                If the answer is not in the context, do not make up answers, just say that you don't know.
                Keep the answer detailed and well formatted based on the information from the context.

                Question:
                {question}

                Context:
                {context}

                Answer:
            """

rag_prompt_template = ChatPromptTemplate.from_template(rag_prompt)

In [48]:
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableLambda
from operator import itemgetter


chatgpt = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

src_rag_response_chain = (
    {
        "context": (itemgetter('context')
                        |
                    RunnableLambda(format_docs)),
        "question": itemgetter("question")
    }
        |
    rag_prompt_template
        |
    chatgpt
        |
    StrOutputParser()
)

rag_chain_w_sources = (
    {
        "context": similarity_retriever,
        "question": RunnablePassthrough()
    }
        |
    RunnablePassthrough.assign(response=src_rag_response_chain)
)

In [49]:
query = "What is machine learning?"
result = rag_chain_w_sources.invoke(query)
result

{'context': [Document(id='1acca6d5-a880-457b-b42e-913ad149a7ca', metadata={'title': 'Machine learning', 'id': '564928', 'page': 1, 'source': 'Wikipedia'}, page_content='Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It is a subfield of computer science. The idea came from work in artificial intelligence. Machine learning explores the study and construction of algorithms which can learn and make predictions on data. Such algorithms follow programmed instructions, but can also make predictions or decisions based on data. They build a model from sample inputs. Machine learning is done where designing and programming explicit algorithms cannot be done. Examples include spam filtering, detection of network intruders or malicious insiders working towards a data breach, optical character recognition (OCR), search engines and computer vision.'),
  Document(id='38e35553-3fa7-449a-ba1f-c95d2a982fee', metadata={'id': '359370', 'tit

In [50]:
from IPython.display import display, Markdown

def display_results(result_obj):
    print('Query:')
    display(Markdown(result_obj['question']))
    print()
    print('Response:')
    display(Markdown(result_obj['response']))
    print('='*50)
    print('Sources:')
    for source in result_obj['context']:
        print('Metadata:', source.metadata)
        print('Content Brief:')
        display(Markdown(source.page_content))
        print()


In [51]:
query = "What is machine learning?"
result = rag_chain_w_sources.invoke(query)
display_results(result)

Query:


What is machine learning?


Response:


Machine learning is a subfield of computer science that provides computers with the ability to learn without being explicitly programmed. The concept was introduced by Arthur Samuel in 1959 and is rooted in artificial intelligence. Machine learning focuses on the study and construction of algorithms that can learn from data and make predictions or decisions based on that data. These algorithms follow programmed instructions but can also adapt and improve their performance by building models from sample inputs.

Machine learning is particularly useful in scenarios where designing and programming explicit algorithms is impractical. Common applications include spam filtering, detecting network intruders or malicious insiders, optical character recognition (OCR), search engines, and computer vision.

Within machine learning, there are different types of learning approaches, such as supervised learning, where a function is inferred from labeled training data. In supervised learning, the system learns to produce correct results based on known outcomes from the training data.

Additionally, deep learning is a specialized area of machine learning that utilizes neural networks, particularly those with multiple layers (known as multi-layer neural networks). Deep learning is effective for complex tasks like speech recognition, image understanding, and handwriting recognition, which are challenging for computers but relatively easy for humans. 

Overall, machine learning represents a significant advancement in the ability of computers to process information and make decisions autonomously, drawing inspiration from biological systems while employing distinct methodologies.

Sources:
Metadata: {'page': 1, 'title': 'Machine learning', 'source': 'Wikipedia', 'id': '564928'}
Content Brief:


Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It is a subfield of computer science. The idea came from work in artificial intelligence. Machine learning explores the study and construction of algorithms which can learn and make predictions on data. Such algorithms follow programmed instructions, but can also make predictions or decisions based on data. They build a model from sample inputs. Machine learning is done where designing and programming explicit algorithms cannot be done. Examples include spam filtering, detection of network intruders or malicious insiders working towards a data breach, optical character recognition (OCR), search engines and computer vision.

... (4 outputs left out)


Metadata: {'source': 'Wikipedia', 'id': '6360', 'title': 'Artificial intelligence', 'page': 1}
Content Brief:


Artificial intelligence (AI) is the ability of a computer program or a machine to think and learn. It is also a field of study which tries to make computers "smart". They work on their own without being encoded with commands. John McCarthy came up with the name "Artificial Intelligence" in 1955. In general use, the term "artificial intelligence" means a programme which mimics human cognition. At least some of the things we associate with other minds, such as learning and problem solving can be done by computers, though not in the same way as we do. Andreas Kaplan and Michael Haenlein define AI as a system’s ability to correctly interpret external data, to learn from such data, and to use those learnings to achieve specific goals and tasks through flexible adaptation. An ideal (perfect) intelligent machine is a flexible agent which perceives its environment and takes actions to maximize its chance of success at some goal or objective. As machines become increasingly capable, mental faculties once thought to require intelligence are removed from the definition. For example, optical character recognition is no longer perceived as an example of "artificial intelligence": it is just a routine technology.


Metadata: {'id': '44742', 'title': 'Artificial neural network', 'source': 'Wikipedia', 'page': 1}
Content Brief:


A neural network (also called an ANN or an artificial neural network) is a sort of computer software, inspired by biological neurons. Biological brains are capable of solving difficult problems, but each neuron is only responsible for solving a very small part of the problem. Similarly, a neural network is made up of cells that work together to produce a desired result, although each individual cell is only responsible for solving a small part of the problem. This is one method for creating artificially intelligent programs. Neural networks are an example of machine learning, where a program can change as it learns to solve a problem. A neural network can be trained and improved with each example, but the larger the neural network, the more examples it needs to perform well—often needing millions or billions of examples in the case of deep learning. There are two ways to think of a neural network. First is like a human brain. Second is like a mathematical equation.

In [52]:
query = "What is the difference between AI, ML and DL?"
result = rag_chain_w_sources.invoke(query)
display_results(result)

Query:


What is the difference between AI, ML and DL?


Response:


The difference between AI, ML, and DL can be summarized as follows:

### Artificial Intelligence (AI)
- **Definition**: AI refers to the ability of a computer program or machine to think and learn, mimicking human cognition. It encompasses a broad range of technologies and applications aimed at making machines "smart."
- **Functionality**: AI systems can interpret external data, learn from it, and use those learnings to achieve specific goals through flexible adaptation. The term has evolved, and tasks once considered AI, like optical character recognition, are now seen as routine technologies.

### Machine Learning (ML)
- **Definition**: ML is a subfield of AI that focuses on the study and construction of algorithms that allow computers to learn from data without being explicitly programmed. It emerged from the broader field of artificial intelligence.
- **Functionality**: ML algorithms build models from sample inputs and can make predictions or decisions based on data. It is particularly useful in scenarios where designing explicit algorithms is impractical, such as spam filtering and computer vision.

### Deep Learning (DL)
- **Definition**: DL is a specialized subset of machine learning that primarily uses neural networks with multiple layers (deep neural networks) to process data. It is often referred to as deep structured learning or hierarchical learning.
- **Functionality**: In deep learning, the information processed becomes more abstract with each added layer, making it particularly effective for complex tasks like speech and image recognition. DL models are inspired by the information processing patterns of biological nervous systems.

In summary, AI is the overarching field that includes both ML and DL. ML is a subset of AI that focuses on learning from data, while DL is a further specialization within ML that utilizes deep neural networks to handle complex data processing tasks.

Sources:
Metadata: {'source': 'Wikipedia', 'page': 1, 'title': 'Deep learning', 'id': '663523'}
Content Brief:


Deep learning (also called deep structured learning or hierarchical learning) is a kind of machine learning, which is mostly used with certain kinds of neural networks. As with other kinds of machine-learning, learning sessions can be unsupervised, semi-supervised, or supervised. In many cases, structures are organised so that there is at least one intermediate layer (or hidden layer), between the input layer and the output layer. Certain tasks, such as as recognizing and understanding speech, images or handwriting, is easy to do for humans. However, for a computer, these tasks are very difficult to do. In a multi-layer neural network (having more than two layers), the information processed will become more abstract with each added layer. Deep learning models are inspired by information processing and communication patterns in biological nervous systems; they are different from the structural and functional properties of biological brains (especially the human brain) in many ways, which make them incompatible with neuroscience evidences.

... (4 outputs left out)


Metadata: {'title': 'Loop AI Labs', 'source': 'Wikipedia', 'id': '669662', 'page': 1}
Content Brief:


Loop AI Labs is an AI and cognitive computing company that focuses on language understanding technology. The company was founded in San Francisco in 2012 by Italian entrepreneur Gianmauro Calafiore, who sold his company Gsmbox to in 2004 and then relocated from Italy to San Francisco. Wanting to start an artificial intelligence company, he recruited two veterans of the project, the largest government-funded AI project in history, who had worked on the project at and Stanford University's . The original company name, "Soshoma", was changed to Loop AI Labs in 2015 after the company decided to change its focus from consumer-oriented to enterprise. Loop AI Labs is headquartered in San Francisco, California, with offices in New York, Milan, and Singapore. The company is privately funded. On May 4, 2017, Loop AI Labs entered into a deal with , a leading European provider of mobile messaging and solutions, to bring their cognitive computing technology to LINK's business clients, which cover 234 million people across Europe.


Metadata: {'title': 'Artificial neural network', 'page': 1, 'id': '44742', 'source': 'Wikipedia'}
Content Brief:


A neural network (also called an ANN or an artificial neural network) is a sort of computer software, inspired by biological neurons. Biological brains are capable of solving difficult problems, but each neuron is only responsible for solving a very small part of the problem. Similarly, a neural network is made up of cells that work together to produce a desired result, although each individual cell is only responsible for solving a small part of the problem. This is one method for creating artificially intelligent programs. Neural networks are an example of machine learning, where a program can change as it learns to solve a problem. A neural network can be trained and improved with each example, but the larger the neural network, the more examples it needs to perform well—often needing millions or billions of examples in the case of deep learning. There are two ways to think of a neural network. First is like a human brain. Second is like a mathematical equation.

In [53]:
query = "What is the difference between transformers and vision transformers?"
result = rag_chain_w_sources.invoke(query)
display_results(result)

Query:


What is the difference between transformers and vision transformers?


Response:


The difference between transformers and vision transformers primarily lies in their application and the way they process input data.

1. **Architecture and Input Handling**:
   - **Transformers**: Originally designed for natural language processing (NLP), transformers process sequences of tokens (words) using self-attention mechanisms. They take a 1D sequence of token embeddings as input, which allows them to capture relationships between words in a sentence effectively.
   - **Vision Transformers (ViT)**: Vision transformers adapt the transformer architecture for image classification tasks by treating image patches as tokens. An image is divided into fixed-size patches, which are then flattened and linearly embedded into a sequence of vectors. This sequence is fed into a standard transformer encoder, allowing the model to learn spatial relationships within the image.

2. **Inductive Bias**:
   - **Transformers**: In NLP, transformers benefit from the sequential nature of text, where the order of tokens is crucial for understanding context and meaning.
   - **Vision Transformers**: ViTs lack some of the inductive biases inherent to convolutional neural networks (CNNs), such as translation equivariance and locality. This means that while CNNs are designed to recognize patterns in spatial hierarchies (like edges and textures), ViTs rely on large-scale training to learn these patterns from the data itself.

3. **Performance and Efficiency**:
   - **Transformers**: In NLP, transformers have become the standard due to their ability to scale and handle large datasets effectively, achieving state-of-the-art results across various tasks.
   - **Vision Transformers**: ViTs have shown competitive performance in image recognition tasks, especially when pre-trained on large datasets. They can outperform traditional CNNs in terms of compute efficiency, requiring significantly less computational resources to achieve similar or better performance on image classification benchmarks.

4. **Attention Mechanism**:
   - **Transformers**: Utilize self-attention to weigh the importance of different tokens in a sequence, allowing for a flexible understanding of context.
   - **Vision Transformers**: Employ a similar self-attention mechanism but apply it across the entire image, enabling the model to integrate information from different parts of the image, even in the early layers of the network.

In summary, while both transformers and vision transformers share a common architectural foundation, they differ significantly in their input processing, application domains, and the types of biases they leverage to achieve performance in their respective tasks.

Sources:
Metadata: {'id': 'd1c5e7ce-82ff-4fda-85d5-f44b0231f052', 'source': './rag_docs/vision_transformer.pdf', 'page': 7, 'title': 'vision_transformer.pdf'}
Content Brief:


Focuses on a controlled scaling study of various models, including ResNets and Vision Transformers, evaluating their transfer performance from the JFT-300M dataset. It highlights the performance versus pre-training cost, revealing that Vision Transformers outperform ResNets in terms of compute efficiency and suggesting potential for further scaling efforts. Additionally, it discusses the performance of hybrid models in comparison to pure Vision Transformers.
Published as a conference paper at ICLR 2021
4.4
SCALING STUDY
We perform a controlled scaling study of different models by evaluating transfer performance from
JFT-300M. In this setting data size does not bottleneck the models’ performances, and we assess
performance versus pre-training cost of each model. The model set includes: 7 ResNets, R50x1,
R50x2 R101x1, R152x1, R152x2, pre-trained for 7 epochs, plus R152x2 and R200x3 pre-trained
for 14 epochs; 6 Vision Transformers, ViT-B/32, B/16, L/32, L/16, pre-trained for 7 epochs, plus
L/16 and H/14 pre-trained for 14 epochs; and 5 hybrids, R50+ViT-B/32, B/16, L/32, L/16 pre-
trained for 7 epochs, plus R50+ViT-L/16 pre-trained for 14 epochs (for hybrids, the number at the
end of the model name stands not for the patch size, but for the total dowsampling ratio in the ResNet
backbone).
Figure 5 contains the transfer performance versus total pre-training compute (see Appendix D.5
for details on computational costs). Detailed results per model are provided in Table 6 in the Ap-
pendix. A few patterns can be observed. First, Vision Transformers dominate ResNets on the
performance/compute trade-off. ViT uses approximately 2 −4× less compute to attain the same
performance (average over 5 datasets). Second, hybrids slightly outperform ViT at small compu-
tational budgets, but the difference vanishes for larger models. This result is somewhat surprising,
since one might expect convolutional local feature processing to assist ViT at any size. Third, Vision
Transformers appear not to saturate within the range tried, motivating future scaling efforts.
4.5
INSPECTING VISION TRANSFORMER
Input
Attention
Figure 6: Representative ex-
amples of attention from the
output token to the input
space. See Appendix D.7 for
details.
To begin to understand how the Vision Transformer processes im-
age data, we analyze its internal representations. The ﬁrst layer of
the Vision Transformer linearly projects the ﬂattened patches into a
lower-dimensional space (Eq. 1). Figure 7 (left) shows the top prin-
cipal components of the the learned embedding ﬁlters. The com-
ponents resemble plausible basis functions for a low-dimensional
representation of the ﬁne structure within each patch.
After the projection, a learned position embedding is added to the
patch representations. Figure 7 (center) shows that the model learns
to encode distance within the image in the similarity of position em-
beddings, i.e. closer patches tend to have more similar position em-
beddings. Further, the row-column structure appears; patches in the
same row/column have similar embeddings. Finally, a sinusoidal
structure is sometimes apparent for larger grids (Appendix D). That
the position embeddings learn to represent 2D image topology ex-
plains why hand-crafted 2D-aware embedding variants do not yield
improvements (Appendix D.4).
Self-attention allows ViT to integrate information across the entire
image even in the lowest layers. We investigate to what degree
the network makes use of this capability. Speciﬁcally, we compute
the average distance in image space across which information is
integrated, based on the attention weights (Figure 7, right). This
“attention distance” is analogous to receptive ﬁeld size in CNNs.
We ﬁnd that some heads attend to most of the image already in the lowest layers, showing that
the ability to integrate information globally is indeed used by the model. Other attention heads

... (4 outputs left out)


Metadata: {'page': 1, 'title': 'vision_transformer.pdf', 'source': './rag_docs/vision_transformer.pdf', 'id': '51a0f506-eb8c-423f-876c-5ea1f04eb870'}
Content Brief:


Focuses on the performance of the Vision Transformer (ViT) in image recognition tasks, highlighting its ability to achieve state-of-the-art results when pre-trained on large datasets. It contrasts the inductive biases of convolutional neural networks (CNNs) with the advantages of large-scale training for ViT, demonstrating its effectiveness on various benchmarks. Additionally, it references related work in the field, particularly models that integrate self-attention with CNNs.
Published as a conference paper at ICLR 2021
inherent to CNNs, such as translation equivariance and locality, and therefore do not generalize well
when trained on insufﬁcient amounts of data.
However, the picture changes if the models are trained on larger datasets (14M-300M images). We
ﬁnd that large scale training trumps inductive bias. Our Vision Transformer (ViT) attains excellent
results when pre-trained at sufﬁcient scale and transferred to tasks with fewer datapoints. When
pre-trained on the public ImageNet-21k dataset or the in-house JFT-300M dataset, ViT approaches
or beats state of the art on multiple image recognition benchmarks. In particular, the best model
reaches the accuracy of 88.55% on ImageNet, 90.72% on ImageNet-ReaL, 94.55% on CIFAR-100,
and 77.63% on the VTAB suite of 19 tasks.
2
RELATED WORK
Transformers were proposed by Vaswani et al. (2017) for machine translation, and have since be-
come the state of the art method in many NLP tasks. Large Transformer-based models are often
pre-trained on large corpora and then ﬁne-tuned for the task at hand: BERT (Devlin et al., 2019)
uses a denoising self-supervised pre-training task, while the GPT line of work uses language mod-
eling as its pre-training task (Radford et al., 2018; 2019; Brown et al., 2020).
Naive application of self-attention to images would require that each pixel attends to every other
pixel. With quadratic cost in the number of pixels, this does not scale to realistic input sizes. Thus,
to apply Transformers in the context of image processing, several approximations have been tried in
the past. Parmar et al. (2018) applied the self-attention only in local neighborhoods for each query
pixel instead of globally. Such local multi-head dot-product self attention blocks can completely
replace convolutions (Hu et al., 2019; Ramachandran et al., 2019; Zhao et al., 2020). In a different
line of work, Sparse Transformers (Child et al., 2019) employ scalable approximations to global self-
attention in order to be applicable to images. An alternative way to scale attention is to apply it in
blocks of varying sizes (Weissenborn et al., 2019), in the extreme case only along individual axes (Ho
et al., 2019; Wang et al., 2020a). Many of these specialized attention architectures demonstrate
promising results on computer vision tasks, but require complex engineering to be implemented
efﬁciently on hardware accelerators.
Most related to ours is the model of Cordonnier et al. (2020), which extracts patches of size 2 × 2
from the input image and applies full self-attention on top. This model is very similar to ViT,
but our work goes further to demonstrate that large scale pre-training makes vanilla transformers
competitive with (or even better than) state-of-the-art CNNs. Moreover, Cordonnier et al. (2020)
use a small patch size of 2 × 2 pixels, which makes the model applicable only to small-resolution
images, while we handle medium-resolution images as well.
There has also been a lot of interest in combining convolutional neural networks (CNNs) with forms
of self-attention, e.g. by augmenting feature maps for image classiﬁcation (Bello et al., 2019) or by
further processing the output of a CNN using self-attention, e.g. for object detection (Hu et al., 2018;
Carion et al., 2020), video processing (Wang et al., 2018; Sun et al., 2019), image classiﬁcation (Wu
et al., 2020), unsupervised object discovery (Locatello et al., 2020), or uniﬁed text-vision tasks (Chen


Metadata: {'title': 'vision_transformer.pdf', 'id': '3ff22f5f-18fe-49ff-b6fe-be9a59fbf74b', 'source': './rag_docs/vision_transformer.pdf', 'page': 9}
Content Brief:


Focuses on the references cited in the research paper, highlighting key works related to Transformers, image recognition, and self-attention mechanisms. These references provide foundational and contemporary insights that support the development and evaluation of the Vision Transformer (ViT) model presented in the paper.
Published as a conference paper at ICLR 2021
Alexei Baevski and Michael Auli. Adaptive input representations for neural language modeling. In
ICLR, 2019.
I. Bello, B. Zoph, Q. Le, A. Vaswani, and J. Shlens. Attention augmented convolutional networks.
In ICCV, 2019.
Lucas Beyer, Olivier J. H´enaff, Alexander Kolesnikov, Xiaohua Zhai, and A¨aron van den Oord. Are
we done with imagenet? arXiv, 2020.
Tom B Brown, Benjamin Mann, Nick Ryder, Melanie Subbiah, Jared Kaplan, Prafulla Dhariwal,
Arvind Neelakantan, Pranav Shyam, Girish Sastry, Amanda Askell, et al. Language models are
few-shot learners. arXiv, 2020.
Nicolas Carion, Francisco Massa, Gabriel Synnaeve, Nicolas Usunier, Alexander Kirillov, and
Sergey Zagoruyko. End-to-end object detection with transformers. In ECCV, 2020.
Mark Chen, Alec Radford, Rewon Child, Jeff Wu, and Heewoo Jun. Generative pretraining from
pixels. In ICML, 2020a.
Ting Chen, Simon Kornblith, Mohammad Norouzi, and Geoffrey E. Hinton. A simple framework
for contrastive learning of visual representations. In ICML, 2020b.
Yen-Chun Chen, Linjie Li, Licheng Yu, Ahmed El Kholy, Faisal Ahmed, Zhe Gan, Yu Cheng, and
Jingjing Liu. UNITER: UNiversal Image-TExt Representation Learning. In ECCV, 2020c.
Rewon Child, Scott Gray, Alec Radford, and Ilya Sutskever. Generating long sequences with sparse
transformers. arXiv, 2019.
Jean-Baptiste Cordonnier, Andreas Loukas, and Martin Jaggi. On the relationship between self-
attention and convolutional layers. In ICLR, 2020.
J. Deng, W. Dong, R. Socher, L. Li, Kai Li, and Li Fei-Fei. Imagenet: A large-scale hierarchical
image database. In CVPR, 2009.
Jacob Devlin, Ming-Wei Chang, Kenton Lee, and Kristina Toutanova. BERT: Pre-training of deep
bidirectional transformers for language understanding. In NAACL, 2019.
Josip Djolonga, Jessica Yung, Michael Tschannen, Rob Romijnders, Lucas Beyer, Alexander
Kolesnikov, Joan Puigcerver, Matthias Minderer, Alexander D’Amour, Dan Moldovan, Sylvan
Gelly, Neil Houlsby, Xiaohua Zhai, and Mario Lucic. On robustness and transferability of convo-
lutional neural networks. arXiv, 2020.
Kaiming He, Xiangyu Zhang, Shaoqing Ren, and Jian Sun. Deep residual learning for image recog-
nition. In CVPR, 2016.
Kaiming He, Haoqi Fan, Yuxin Wu, Saining Xie, and Ross Girshick.
Momentum contrast for
unsupervised visual representation learning. In CVPR, 2020.
Jonathan Ho, Nal Kalchbrenner, Dirk Weissenborn, and Tim Salimans. Axial attention in multidi-
mensional transformers. arXiv, 2019.
Han Hu, Jiayuan Gu, Zheng Zhang, Jifeng Dai, and Yichen Wei. Relation networks for object
detection. In CVPR, 2018.
Han Hu, Zheng Zhang, Zhenda Xie, and Stephen Lin. Local relation networks for image recognition.
In ICCV, 2019.
Zilong Huang, Xinggang Wang, Yunchao Wei, Lichao Huang, Humphrey Shi, Wenyu Liu, and
Thomas S. Huang. Ccnet: Criss-cross attention for semantic segmentation. In ICCV, 2020.
Olivier J. H´enaff, Aravind Srinivas, Jeffrey De Fauw, Ali Razavi, Carl Doersch, S. M. Ali Eslami,
and Aaron van den Oord. Data-efﬁcient image recognition with contrastive predictive coding. In
ICML, 2020.
10

In [54]:
query = "What is an Agentic AI System?"
result = rag_chain_w_sources.invoke(query)
display_results(result)

Query:


What is an Agentic AI System?


Response:


The context provided does not contain specific information about an "Agentic AI System." Therefore, I don't know what an Agentic AI System is.

Sources:
Metadata: {'page': 1, 'title': 'Artificial intelligence', 'source': 'Wikipedia', 'id': '6360'}
Content Brief:


Artificial intelligence (AI) is the ability of a computer program or a machine to think and learn. It is also a field of study which tries to make computers "smart". They work on their own without being encoded with commands. John McCarthy came up with the name "Artificial Intelligence" in 1955. In general use, the term "artificial intelligence" means a programme which mimics human cognition. At least some of the things we associate with other minds, such as learning and problem solving can be done by computers, though not in the same way as we do. Andreas Kaplan and Michael Haenlein define AI as a system’s ability to correctly interpret external data, to learn from such data, and to use those learnings to achieve specific goals and tasks through flexible adaptation. An ideal (perfect) intelligent machine is a flexible agent which perceives its environment and takes actions to maximize its chance of success at some goal or objective. As machines become increasingly capable, mental faculties once thought to require intelligence are removed from the definition. For example, optical character recognition is no longer perceived as an example of "artificial intelligence": it is just a routine technology.

... (4 outputs left out)


Metadata: {'title': 'Shakey the robot', 'source': 'Wikipedia', 'id': '692745', 'page': 1}
Content Brief:


Shakey the Robot was the first general purpose mobile AI robot. The project combined research in robotics, computer vision, and natural language processing. Because of this, it was the first project that melded logical reasoning and physical action. Shakey was developed at the Artificial Intelligence Center of Stanford Research Institute (now called SRI International) by Nils John Nilsson from 1966 to 1972.


Metadata: {'page': 1, 'id': '564218', 'source': 'Wikipedia', 'title': 'Robot lawyer'}
Content Brief:


A robot lawyer is an artificial intelligence (AI) computer program. It is designed to ask the same questions as a real lawyer about certain legal issues. Robot lawyers are being used in many countries around the world including the United States, the United Kingdom, and Holland.

# Build a RAG System with Source Citations Agentic Pipeline

In [55]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = """You are an assistant who is an expert in question-answering tasks.
                Answer the following question using only the following pieces of retrieved context.
                If the answer is not in the context, do not make up answers, just say that you don't know.
                Keep the answer detailed and well formatted based on the information from the context.

                Question:
                {question}

                Context:
                {context}

                Answer:
            """

rag_prompt_template = ChatPromptTemplate.from_template(rag_prompt)
rag_prompt_template.pretty_print()

================================ Human Message =================================

You are an assistant who is an expert in question-answering tasks.
                Answer the following question using only the following pieces of retrieved context.
                If the answer is not in the context, do not make up answers, just say that you don't know.
                Keep the answer detailed and well formatted based on the information from the context.

                Question:
                {question}

                Context:
                {context}

                Answer:
            


In [56]:
citations_prompt = """You are an assistant who is an expert in analyzing answers to questions
                      and finding out referenced citations from context articles.

                      Given the following question, context and generated answer,
                      analyze the generated answer and quote citations from context articles
                      that can be used to justify the generated answer.

                      Question:
                      {question}

                      Context Articles:
                      {context}

                      Answer:
                      {answer}
                  """

cite_prompt_template = ChatPromptTemplate.from_template(citations_prompt)
cite_prompt_template.pretty_print()

================================ Human Message =================================

You are an assistant who is an expert in analyzing answers to questions
                      and finding out referenced citations from context articles.

                      Given the following question, context and generated answer,
                      analyze the generated answer and quote citations from context articles
                      that can be used to justify the generated answer.

                      Question:
                      {question}

                      Context Articles:
                      {context}

                      Answer:
                      {answer}
                  


In [57]:
from pydantic import BaseModel, Field
from typing import List

class Citation(BaseModel):
    id: str = Field(description="""The string ID of a SPECIFIC context article
                                   which justifies the answer.""")
    source: str = Field(description="""The source of the SPECIFIC context article
                                       which justifies the answer.""")
    title: str = Field(description="""The title of the SPECIFIC context article
                                      which justifies the answer.""")
    page: int = Field(description="""The page number of the SPECIFIC context article
                                     which justifies the answer.""")
    quotes: str = Field(description="""The VERBATIM sentences from the SPECIFIC context article
                                      that are used to generate the answer.
                                      Should be exact sentences from context article without missing words.""")


class QuotedCitations(BaseModel):
    """Quote citations from given context articles
       that can be used to justify the generated answer. Can be multiple articles."""
    citations: List[Citation] = Field(description="""Citations (can be multiple) from the given
                                                     context articles that justify the answer.""")

In [58]:
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableLambda
from operator import itemgetter


chatgpt = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)
structured_chatgpt = chatgpt.with_structured_output(QuotedCitations)


def format_docs_with_metadata(docs: List[Document]) -> str:
    formatted_docs = [
        f"""Context Article ID: {doc.metadata['id']}
            Context Article Source: {doc.metadata['source']}
            Context Article Title: {doc.metadata['title']}
            Context Article Page: {doc.metadata['page']}
            Context Article Details: {doc.page_content}
         """
            for i, doc in enumerate(docs)
    ]
    return "\n\n" + "\n\n".join(formatted_docs)

rag_response_chain = (
    {
        "context": (itemgetter('context')
                        |
                    RunnableLambda(format_docs_with_metadata)),
        "question": itemgetter("question")
    }
        |
    rag_prompt_template
        |
    chatgpt
        |
    StrOutputParser()
)

cite_response_chain = (
    {
        "context": itemgetter('context'),
        "question": itemgetter("question"),
        "answer": itemgetter("answer")
    }
        |
    cite_prompt_template
        |
    structured_chatgpt
)

rag_chain_w_citations = (
    {
        "context": similarity_retriever,
        "question": RunnablePassthrough()
    }
        |
    RunnablePassthrough.assign(answer=rag_response_chain)
        |
    RunnablePassthrough.assign(citations=cite_response_chain)
)

In [59]:
query = "What is machine learning"
result = rag_chain_w_citations.invoke(query)
result

{'context': [Document(id='1acca6d5-a880-457b-b42e-913ad149a7ca', metadata={'title': 'Machine learning', 'id': '564928', 'source': 'Wikipedia', 'page': 1}, page_content='Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It is a subfield of computer science. The idea came from work in artificial intelligence. Machine learning explores the study and construction of algorithms which can learn and make predictions on data. Such algorithms follow programmed instructions, but can also make predictions or decisions based on data. They build a model from sample inputs. Machine learning is done where designing and programming explicit algorithms cannot be done. Examples include spam filtering, detection of network intruders or malicious insiders working towards a data breach, optical character recognition (OCR), search engines and computer vision.'),
  Document(id='38e35553-3fa7-449a-ba1f-c95d2a982fee', metadata={'id': '359370', 'pag

In [60]:
result['citations'].dict()['citations']

/tmp/ipykernel_593/3486563740.py:1: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  result['citations'].dict()['citations']


[{'id': '564928',
  'source': 'Wikipedia',
  'title': 'Machine learning',
  'page': 1,
  'quotes': 'Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It is a subfield of computer science. The idea came from work in artificial intelligence. Machine learning explores the study and construction of algorithms which can learn and make predictions on data. Such algorithms follow programmed instructions, but can also make predictions or decisions based on data. They build a model from sample inputs. Machine learning is done where designing and programming explicit algorithms cannot be done. Examples include spam filtering, detection of network intruders or malicious insiders working towards a data breach, optical character recognition (OCR), search engines and computer vision.'},
 {'id': '6360',
  'source': 'Wikipedia',
  'title': 'Artificial intelligence',
  'page': 1,
  'quotes': 'Artificial intelligence (AI) is the ability of a

In [61]:
import re
# used mostly for nice display formatting, ignore if not needed
def get_cited_context(result_obj):
    # Dictionary to hold separate citation information for each unique source and title combination
    source_with_citations = {}

    def highlight_text(context, quote):
        # Normalize whitespace and remove unnecessary punctuation
        quote = re.sub(r'\s+', ' ', quote).strip()
        context = re.sub(r'\s+', ' ', context).strip()

        # Split quote into phrases, being careful with punctuation
        phrases = [phrase.strip() for phrase in re.split(r'[.!?]', quote) if phrase.strip()]

        highlighted_context = context

        for phrase in phrases: # for each quoted phrase

            # Create regex pattern to match cited phrases
            # Escape special regex characters, but preserve word boundaries
            escaped_phrase = re.escape(phrase)
            # Create regex pattern that allows for slight variations
            pattern = re.compile(r'\b' + escaped_phrase + r'\b', re.IGNORECASE)

            # Replace all matched phrases with bolded version
            highlighted_context = pattern.sub(lambda m: f"**{m.group(0)}**", highlighted_context)

        return highlighted_context

    # Process the citation data
    for cite in result_obj['citations'].dict()['citations']:
        cite_id = cite['id']
        title = cite['title']
        source = cite['source']
        page = cite['page']
        quote = cite['quotes']

        # Check if the (source, title) key exists, and initialize if it doesn't
        if (source, title) not in source_with_citations:
            source_with_citations[(source, title)] = {
                'title': title,
                'source': source,
                'citations': []
            }

        # Find or create the citation entry for this unique (id, page) combination
        citation_entry = next(
            (c for c in source_with_citations[(source, title)]['citations'] if c['id'] == cite_id and c['page'] == page),
            None
        )
        if citation_entry is None:
            citation_entry = {'id': cite_id, 'page': page, 'quote': [quote], 'context': None}
            source_with_citations[(source, title)]['citations'].append(citation_entry)
        else:
            citation_entry['quote'].append(quote)

    # Process context data
    for context in result_obj['context']:
        context_id = context.metadata['id']
        context_page = context.metadata['page']
        source = context.metadata['source']
        title = context.metadata['title']
        page_content = context.page_content

        # Match the context to the correct citation entry by source, title, id, and page
        if (source, title) in source_with_citations:
            for citation in source_with_citations[(source, title)]['citations']:
                if citation['id'] == context_id and citation['page'] == context_page:
                    # Apply highlighting for each quote in the citation's quote list
                    highlighted_content = page_content
                    for quote in citation['quote']:
                        highlighted_content = highlight_text(highlighted_content, quote)
                    citation['context'] = highlighted_content

    # Convert the dictionary to a list of dictionaries for separate entries
    final_result_list = [
        {
            'title': details['title'],
            'source': details['source'],
            'citations': details['citations']
        }
        for details in source_with_citations.values()
    ]

    return final_result_list


In [62]:
get_cited_context(result)

/tmp/ipykernel_593/3913888139.py:31: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  for cite in result_obj['citations'].dict()['citations']:


[{'title': 'Machine learning',
  'source': 'Wikipedia',
  'citations': [{'id': '564928',
    'page': 1,
    'quote': ['Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It is a subfield of computer science. The idea came from work in artificial intelligence. Machine learning explores the study and construction of algorithms which can learn and make predictions on data. Such algorithms follow programmed instructions, but can also make predictions or decisions based on data. They build a model from sample inputs. Machine learning is done where designing and programming explicit algorithms cannot be done. Examples include spam filtering, detection of network intruders or malicious insiders working towards a data breach, optical character recognition (OCR), search engines and computer vision.'],
    'context': 'Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). **It 

In [63]:
from IPython.display import display, Markdown

def display_results(result_obj):
    print('Query:')
    display(Markdown(result_obj['question']))
    print()
    print('Response:')
    display(Markdown(result_obj['answer']))
    print('='*50)
    print('Sources:')
    cited_context = get_cited_context(result_obj)
    for source in cited_context:
        print('Title:', source['title'], ' ', 'Source:', source['source'])
        print('Citations:')
        for citation in source['citations']:
            print('ID:', citation['id'], ' ', 'Page:', citation['page'])
            print('Cited Quotes:')
            display(Markdown('*'+' '.join(citation['quote'])+'*'))
            print('Cited Context:')
            display(Markdown(citation['context']))
            print()


In [64]:
display_results(result)

Query:


What is machine learning


Response:


Machine learning is a subfield of computer science that provides computers with the ability to learn without being explicitly programmed. The concept was introduced by Arthur Samuel in 1959 and is rooted in the broader field of artificial intelligence (AI). 

Machine learning focuses on the study and construction of algorithms that can learn from and make predictions based on data. These algorithms operate by following programmed instructions while also being capable of making predictions or decisions based on the data they process. They build models from sample inputs, which allows them to function in scenarios where traditional programming methods are insufficient. 

Some common applications of machine learning include:
- Spam filtering
- Detection of network intruders or malicious insiders
- Optical character recognition (OCR)
- Search engines
- Computer vision

Overall, machine learning enables systems to improve their performance on tasks over time as they are exposed to more data, making it a powerful tool in various technological applications.

Sources:
Title: Machine learning   Source: Wikipedia
Citations:
ID: 564928   Page: 1
Cited Quotes:


/tmp/ipykernel_593/3913888139.py:31: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  for cite in result_obj['citations'].dict()['citations']:


... (3 outputs left out)


Title: Artificial intelligence   Source: Wikipedia
Citations:
ID: 6360   Page: 1
Cited Quotes:


*Artificial intelligence (AI) is the ability of a computer program or a machine to think and learn. It is also a field of study which tries to make computers "smart". They work on their own without being encoded with commands. John McCarthy came up with the name "Artificial Intelligence" in 1955. In general use, the term "artificial intelligence" means a programme which mimics human cognition.*

Cited Context:


**Artificial intelligence (AI) is the ability of a computer program or a machine to think and learn**. It is also a field of study which tries to make computers "smart". **They work on their own without being encoded with commands**. **John McCarthy came up with the name "Artificial Intelligence" in 1955**. **In general use, the term "artificial intelligence" means a programme which mimics human cognition**. At least some of the things we associate with other minds, such as learning and problem solving can be done by computers, though not in the same way as we do. Andreas Kaplan and Michael Haenlein define AI as a system’s ability to correctly interpret external data, to learn from such data, and to use those learnings to achieve specific goals and tasks through flexible adaptation. An ideal (perfect) intelligent machine is a flexible agent which perceives its environment and takes actions to maximize its chance of success at some goal or objective. As machines become increasingly capable, mental faculties once thought to require intelligence are removed from the definition. For example, optical character recognition is no longer perceived as an example of "artificial intelligence": it is just a routine technology.

In [65]:
query = "What is AI, ML and DL?"
result = rag_chain_w_citations.invoke(query)
display_results(result)

Query:


What is AI, ML and DL?


Response:


**Artificial Intelligence (AI)**: AI is defined as the ability of a computer program or machine to think and learn. It is also a field of study aimed at making computers "smart," allowing them to operate independently without being explicitly programmed with commands. The term was coined by John McCarthy in 1955. AI encompasses systems that can interpret external data, learn from it, and adapt to achieve specific goals. As technology advances, tasks once considered to require intelligence, such as optical character recognition, are no longer classified as AI but rather as routine technologies.

**Machine Learning (ML)**: ML is a subfield of computer science that provides computers the ability to learn without being explicitly programmed. This concept emerged from the broader field of artificial intelligence. Machine learning focuses on the development of algorithms that can learn from and make predictions based on data. These algorithms can follow programmed instructions while also making decisions based on the data they process. ML is particularly useful in scenarios where traditional programming is impractical, such as spam filtering, network intrusion detection, and computer vision.

**Deep Learning (DL)**: DL is a specialized form of machine learning that primarily utilizes neural networks, particularly those with multiple layers (known as multi-layer neural networks). It involves learning sessions that can be unsupervised, semi-supervised, or supervised. Deep learning excels in tasks that are challenging for computers, such as speech recognition, image understanding, and handwriting recognition. The architecture of deep learning models is inspired by the information processing patterns found in biological nervous systems, although they differ significantly from the structural and functional properties of human brains.

Sources:
Title: Artificial intelligence   Source: Wikipedia
Citations:
ID: 6360   Page: 1
Cited Quotes:


/tmp/ipykernel_593/3913888139.py:31: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  for cite in result_obj['citations'].dict()['citations']:


... (7 outputs left out)


Title: Deep learning   Source: Wikipedia
Citations:
ID: 663523   Page: 1
Cited Quotes:


*Deep learning (also called deep structured learning or hierarchical learning) is a kind of machine learning, which is mostly used with certain kinds of neural networks. As with other kinds of machine-learning, learning sessions can be unsupervised, semi-supervised, or supervised. In many cases, structures are organised so that there is at least one intermediate layer (or hidden layer), between the input layer and the output layer.*

Cited Context:


**Deep learning (also called deep structured learning or hierarchical learning) is a kind of machine learning, which is mostly used with certain kinds of neural networks**. **As with other kinds of machine-learning, learning sessions can be unsupervised, semi-supervised, or supervised**. **In many cases, structures are organised so that there is at least one intermediate layer (or hidden layer), between the input layer and the output layer**. Certain tasks, such as as recognizing and understanding speech, images or handwriting, is easy to do for humans. However, for a computer, these tasks are very difficult to do. In a multi-layer neural network (having more than two layers), the information processed will become more abstract with each added layer. Deep learning models are inspired by information processing and communication patterns in biological nervous systems; they are different from the structural and functional properties of biological brains (especially the human brain) in many ways, which make them incompatible with neuroscience evidences.

In [66]:
query = "How is Machine learning related to supervised learning and clustering?"
result = rag_chain_w_citations.invoke(query)
display_results(result)

Query:


How is Machine learning related to supervised learning and clustering?


Response:


Machine learning is a broad field that encompasses various techniques and methodologies for enabling computers to learn from data. Two important concepts within machine learning are supervised learning and clustering.

### Supervised Learning
- **Definition**: Supervised learning is a specific type of machine learning where the model is trained on labeled data. This means that the training dataset includes both the input data and the corresponding correct outputs (labels).
- **Process**: The system infers a function from this labeled training data, learning how to map inputs to the correct outputs. The goal is to create a classifier that can make predictions on new, unseen data based on the patterns learned from the training data.
- **Inductive Reasoning**: Supervised learning typically employs inductive reasoning to generalize from the training data to make predictions.

### Clustering
- **Definition**: Clustering, or cluster analysis, is a type of data analysis that involves grouping a set of objects in such a way that objects in the same group (or cluster) are more similar to each other than to those in other groups.
- **Application**: Clustering is commonly used in data mining and is a form of unsupervised learning, where the model does not rely on labeled outputs. Instead, it identifies patterns and structures in the data based solely on the input features.

### Relationship Between Machine Learning, Supervised Learning, and Clustering
- **Machine Learning as an Umbrella**: Machine learning serves as the overarching field that includes various learning paradigms, including both supervised learning and clustering.
- **Supervised vs. Unsupervised**: While supervised learning focuses on learning from labeled data to make predictions, clustering operates without labeled outputs, aiming to discover inherent groupings within the data.
- **Different Goals**: The primary goal of supervised learning is to predict outcomes based on input data, whereas clustering aims to identify and group similar data points.

In summary, machine learning encompasses both supervised learning and clustering, with supervised learning focusing on labeled data for prediction and clustering focusing on grouping similar data without labels.

Sources:
Title: Supervised learning   Source: Wikipedia
Citations:
ID: 359370   Page: 1
Cited Quotes:


/tmp/ipykernel_593/3913888139.py:31: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  for cite in result_obj['citations'].dict()['citations']:


... (7 outputs left out)


Title: Cluster analysis   Source: Wikipedia
Citations:
ID: 593732   Page: 1
Cited Quotes:


*Clustering or cluster analysis is a type of data analysis. The analyst groups objects so that objects in the same group (called a cluster) are more similar to each other than to objects in other groups (clusters) in some way.*

Cited Context:


**Clustering or cluster analysis is a type of data analysis**. **The analyst groups objects so that objects in the same group (called a cluster) are more similar to each other than to objects in other groups (clusters) in some way**. This is a common task in data mining.

In [67]:
query = "What is the difference between transformers and vision transformers?"
result = rag_chain_w_citations.invoke(query)
display_results(result)

Query:


What is the difference between transformers and vision transformers?


Response:


The difference between transformers and vision transformers primarily lies in their application and the way they process input data.

1. **Transformers**:
   - Originally designed for natural language processing (NLP) tasks, transformers utilize self-attention mechanisms to process sequences of tokens (words).
   - In NLP, the input is typically a sequence of words, and the transformer architecture processes these tokens to understand context and relationships between them.
   - Transformers have become the standard model for various NLP tasks due to their ability to scale and handle large datasets effectively.

2. **Vision Transformers (ViT)**:
   - Vision Transformers adapt the standard transformer architecture for image classification tasks by treating image patches as tokens.
   - Instead of processing sequences of words, ViTs split an image into fixed-size patches, flatten these patches, and then embed them into a sequence of vectors that can be fed into a transformer encoder.
   - ViTs incorporate position embeddings to retain spatial information about the patches, which is crucial for understanding the structure of images.
   - The architecture allows ViTs to leverage the self-attention mechanism to integrate information across the entire image, enabling them to capture global context effectively.

In summary, while both transformers and vision transformers utilize the same underlying architecture, their key difference lies in the type of data they process (text vs. image) and how they handle that data (tokens vs. image patches). Vision transformers represent a significant shift in applying transformer models directly to image data, demonstrating competitive performance in image recognition tasks when pre-trained on large datasets.

Sources:
Title: vision_transformer.pdf   Source: ./rag_docs/vision_transformer.pdf
Citations:
ID: 5f9f481a-ccd7-4aac-9da7-ff4b981825f0   Page: 0
Cited Quotes:


/tmp/ipykernel_593/3913888139.py:31: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  for cite in result_obj['citations'].dict()['citations']:


... (11 outputs left out)


ID: e4a576a8-0bee-480e-917c-110f474a85ae   Page: 9
Cited Quotes:


*Most related to ours is the model of Cordonnier et al. (2020), which extracts patches of size 2 × 2 from the input image and applies full self-attention on top. This model is very similar to ViT, but our work goes further to demonstrate that large scale pre-training makes vanilla transformers competitive with (or even better than) state-of-the-art CNNs.*

Cited Context:


<IPython.core.display.Markdown object>